In [1]:
from RUN_ALL_LIB import *
import warnings
import gc
import torch
#warnings.filterwarnings("ignore", category=DeprecationWarning)
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import logging
#logging.getLogger("transformers").setLevel(logging.ERROR)
import warnings
#warnings.filterwarnings("ignore", message="Setting `pad_token_id` to `eos_token_id`")

In [ ]:
#ruslanmv/Medical-Llama3-8B
#ruslandev/llama-3-8b-gpt-4o-ru1.0
#meta-llama/Llama-3.1-8B-Instruct
#"meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_4_optim_adamw_torch_fused_r_64_alpha_128_lr_0.0002_drop_out_0.05_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head"
#meta-llama_Llama-3.1-8B-Instruct_lora_epochs_10_batch_3_optim_adamw_torch_fused_r_16_alpha_32_q_proj_v_proj_k_proj_o_proj_gate_proj_up_proj_down_proj_embed_tokens_lm_head


In [6]:
# Обучение без учителя на собранном корпусе dataset.txt
# test 0
run_unsupervised_lora(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    corpus_path="dataset.txt",               # твой файл с текстами
    token=os.getenv("HF_TOKEN"),
    quantization_config=None,                # можно "4bit" если мало памяти
    num_epoch=3,
    batch_size=2,
    grad_accum_steps=4,
    lr=2e-4,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    max_seq_length=512,                      # подбери под свою видеопамять
    chunk_size_sym=1500,                     # кусок текста ~500 токенов
    overlap_sym=150,
    val_split=0.05,
    optim="adamw_torch_fused",
    test_status=False
)


===== UNSUPERVISED LoRA (LM) on dataset.txt =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Epoch,Training Loss,Validation Loss
1,1.637053,1.646770
2,1.480192,1.610837
3,1.405583,1.618318


Training Loss,Validation Loss,Epoch
1.405583,1.610837,3


Validation Perplexity: 5.01
Не удалось записать метрики в Excel (функция может требовать дополнительных параметров)
[after Unsupervised LoRA] allocated=16.2MB reserved=36.0MB max_alloc=25599.3MB


{'perplexity': 5.006999053480134,
 'lora_dir': 'unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_16_alpha_32_lr_0.0002_final'}

In [2]:
# TEST 1 — более лёгкий вариант по памяти
run_unsupervised_lora(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    corpus_path="dataset.txt",
    token=os.getenv("HF_TOKEN"),
    quantization_config="4bit",
    num_epoch=5,
    batch_size=1,
    grad_accum_steps=8,
    lr=2e-4,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    max_seq_length=512,
    chunk_size_sym=1500,
    overlap_sym=150,
    val_split=0.05,
    optim="adamw_torch_fused",
    test_status=False
)


===== UNSUPERVISED LoRA (LM) on dataset.txt =====


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/1481 [00:00<?, ? examples/s]

Map:   0%|          | 0/78 [00:00<?, ? examples/s]

...

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Epoch,Training Loss,Validation Loss
1,1.678761,1.708118
2,1.561634,1.658065
3,1.408633,1.656700
4,1.242284,1.694118
5,1.199667,1.728603


Training Loss,Validation Loss,Epoch
1.199667,1.656700,5


Validation Perplexity: 5.24
Не удалось записать метрики в Excel (функция может требовать дополнительных параметров)
[after Unsupervised LoRA] allocated=16.2MB reserved=336.0MB max_alloc=11023.2MB


{'perplexity': 5.241982553333415,
 'lora_dir': 'unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_5_r_16_alpha_32_lr_0.0002_final'}

In [2]:
# TEST 2 — сильнее LoRA, больше обучаемых параметров
run_unsupervised_lora(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    corpus_path="dataset.txt",
    token=os.getenv("HF_TOKEN"),
    quantization_config=None,
    num_epoch=3,
    batch_size=2,
    grad_accum_steps=4,
    lr=1e-4,
    lora_r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    max_seq_length=512,
    chunk_size_sym=1500,
    overlap_sym=150,
    val_split=0.05,
    optim="adamw_torch_fused",
    test_status=False
)


===== UNSUPERVISED LoRA (LM) on dataset.txt =====


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/1355 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

...

trainable params: 27,262,976 || all params: 8,057,524,224 || trainable%: 0.3384


Epoch,Training Loss,Validation Loss
1,1.644353,1.653635
2,1.505487,1.620084
3,1.455291,1.621925


Training Loss,Validation Loss,Epoch
1.455291,1.620084,3


Validation Perplexity: 5.05
Не удалось записать метрики в Excel (функция может требовать дополнительных параметров)
[after Unsupervised LoRA] allocated=16.2MB reserved=36.0MB max_alloc=23610.7MB


{'perplexity': 5.053516237949994,
 'lora_dir': 'unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_32_alpha_64_lr_0.0001_final'}

In [5]:
# TEST 3 — длиннее контекст
run_unsupervised_lora(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    corpus_path="dataset.txt",
    token=os.getenv("HF_TOKEN"),
    quantization_config=None,
    num_epoch=3,
    batch_size=1,
    grad_accum_steps=8,
    lr=2e-4,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    max_seq_length=1024,
    chunk_size_sym=3000,
    overlap_sym=300,
    val_split=0.05,
    optim="adamw_torch_fused",
    test_status=False
)


===== UNSUPERVISED LoRA (LM) on dataset.txt =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/678 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Epoch,Training Loss,Validation Loss
1,1.583019,1.586096
2,1.493438,1.556771
3,1.381022,1.559242


Training Loss,Validation Loss,Epoch
1.381022,1.556771,3


Validation Perplexity: 4.74
Не удалось записать метрики в Excel (функция может требовать дополнительных параметров)
[after Unsupervised LoRA] allocated=16.2MB reserved=36.0MB max_alloc=25599.3MB


{'perplexity': 4.7434794175365385,
 'lora_dir': 'unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_3_r_16_alpha_32_lr_0.0002_final'}

In [4]:
# TEST 4 — больше эпох, но меньше learning rate
run_unsupervised_lora(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    corpus_path="dataset.txt",
    token=os.getenv("HF_TOKEN"),
    quantization_config=None,
    num_epoch=5,
    batch_size=2,
    grad_accum_steps=4,
    lr=5e-5,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    max_seq_length=512,
    chunk_size_sym=1500,
    overlap_sym=200,
    val_split=0.05,
    optim="adamw_torch_fused",
    test_status=False
)


===== UNSUPERVISED LoRA (LM) on dataset.txt =====


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Map:   0%|          | 0/1407 [00:00<?, ? examples/s]

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


Epoch,Training Loss,Validation Loss
1,1.677504,1.675241
2,1.637076,1.644269
3,1.543726,1.631194
4,1.526995,1.630170
5,1.525703,1.631519


Training Loss,Validation Loss,Epoch
1.525703,1.630170,5


Validation Perplexity: 5.10
Не удалось записать метрики в Excel (функция может требовать дополнительных параметров)
[after Unsupervised LoRA] allocated=16.2MB reserved=36.0MB max_alloc=25599.3MB


{'perplexity': 5.10474056249829,
 'lora_dir': 'unsup_meta-llama_Llama-3.1-8B-Instruct_epochs_5_r_16_alpha_32_lr_5e-05_final'}